# 01 — Data and future-event context

This notebook explains the data boundary, independently constructed outcome, and trailing-only feature design for the Continuous FMEA Risk Monitoring prototype. It uses **development trajectories only** for target/feature demonstrations. It does not inspect official-test labels, fit a model, select a threshold, or participate in the reproducible build.

NASA C-MAPSS FD001 is simulated benchmark data, not field maintenance evidence. Its end-of-life timing is a condition-modeling outcome; it is not an FMEA failure-mode label.

## Reproducibility and rights

Run `python run_analysis.py` from the repository root before using the notebooks. That command obtains or loads the verified private FD001 cache outside the source candidate and generates the authoritative reports. Raw and row-level NASA-derived data must not be redistributed while `REDISTRIBUTION_STATUS = VERIFY_BEFORE_PUBLICATION`.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd


def find_development_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise RuntimeError('Open this notebook from inside the repository tree.')


DEV_ROOT = find_development_root()
PROJECT_ROOT = DEV_ROOT.parent
DATA_DIR = Path(
    os.environ.get(
        'CMAPSS_DATA_DIR',
        str(PROJECT_ROOT / '05_OUTPUTS' / 'private_data_cache' / 'FD001'),
    )
).expanduser().resolve()
sys.path.insert(0, str(DEV_ROOT))

if not DATA_DIR.is_dir():
    raise FileNotFoundError('Run `python run_analysis.py` once to create the verified local data cache.')

DEV_ROOT

In [ ]:
from src.data_loader import UNIT_COLUMN, load_cmapss
from src.data_validation import validate_cmapss_frame
from src.target import (
    PREDICTION_HORIZON,
    RUL_COLUMN,
    TARGET_COLUMN,
    construct_train_targets,
)

cmapss = load_cmapss(DATA_DIR, subset='FD001')
development = construct_train_targets(cmapss.train, horizon=PREDICTION_HORIZON)
development_quality = validate_cmapss_frame(development, target_column=TARGET_COLUMN)
official_test_structure = validate_cmapss_frame(cmapss.test)

pd.DataFrame(
    [
        {
            'partition': 'development',
            'assets': development_quality.number_of_assets,
            'rows': development_quality.number_of_rows,
            'missing_cells': development_quality.missing_cells,
            'duplicate_asset_cycles': development_quality.duplicate_asset_cycles,
            'target_prevalence': development_quality.target_prevalence,
        },
        {
            'partition': 'official test (structure only)',
            'assets': official_test_structure.number_of_assets,
            'rows': official_test_structure.number_of_rows,
            'missing_cells': official_test_structure.missing_cells,
            'duplicate_asset_cycles': official_test_structure.duplicate_asset_cycles,
            'target_prevalence': None,
        },
    ]
)

## Independent target

At asset cycle $t$, the label is one when simulated end of useful life occurs within the next 30 cycles (inclusive):

$$Y(i,t;30)=\mathbb{1}[0 \leq event\_cycle_i-t \leq 30].$$

RUL and event cycle are used only to construct the label and later event metrics. They are prohibited predictors. The final rows below make the inclusive horizon boundary visible on one development engine.

In [ ]:
example_engine = int(development[UNIT_COLUMN].min())
target_context = development.loc[
    development[UNIT_COLUMN].eq(example_engine),
    [UNIT_COLUMN, 'cycle', RUL_COLUMN, TARGET_COLUMN],
].tail(PREDICTION_HORIZON + 5)
target_context

## Parsimonious, trailing-only predictors

The feature module retains age, current operating settings, selected current sensors, 5/10-cycle trailing means/standard deviations/slopes for six sensors, and deviations from each asset's initial reading. A feature at cycle $t$ uses rows at or before $t$; windows are never centered. Sensor selection is descriptive and must not be interpreted causally.

In [ ]:
from src.features import (
    FeatureSpec,
    assert_future_perturbation_invariance,
    build_feature_frame,
    feature_columns,
)
from src.target import assert_predictors_are_leakage_safe

spec = FeatureSpec()
feature_frame = build_feature_frame(cmapss.train, spec)
predictors = feature_columns(feature_frame)
assert_predictors_are_leakage_safe(predictors)

family_counts = pd.Series(
    {
        'age': sum(name == 'asset_age' for name in predictors),
        'operating_settings': sum(name.startswith('current__op_setting_') for name in predictors),
        'current_sensors': sum(name.startswith('current__sensor_') for name in predictors),
        'initial_deviation': sum(name.endswith('__from_initial') for name in predictors),
        'trailing_statistics': sum('__roll_' in name for name in predictors),
    },
    name='feature_count',
)
family_counts.to_frame()

In [ ]:
# Stress test: alter only future sensor rows for one development engine.
# Earlier-time features must remain bit-for-bit unchanged within tolerance.
perturbation = assert_future_perturbation_invariance(
    cmapss.train,
    asset_id=example_engine,
    after_cycle=20,
    spec=spec,
)
perturbation

## Interpretation boundary

Passing this demonstration supports the timing implementation; it does not establish field validity. The authoritative quality report, test-access log, and predictive results are generated by `run_analysis.py`. Continue with notebook 02 only after that frozen run exists.